# Predicting Residential EV Charging Loads using Neural Networks

Build a neural network in PyTorch that predicts the energy (kWh) drawn during EV charging sessions
at residential apartment buildings in Norway. You'll clean the data, add local traffic counts as
extra features, fit a linear-regression baseline, and compare it to the network.

The data files are already in `datasets/` (`EV charging reports.csv`, `Local traffic distribution.csv`)
and `models/model4500.pth` is in `models/`.

### Heads-up on the raw data (read before starting)
These are the raw Norway/Mendeley files, so a few things differ from the clean task wording. You'll
hit these as you go -- they're hints, not solutions:
- **Both CSVs are semicolon-delimited** (`;`), and numbers use **commas as decimals** (`0,3`). So
  `pd.read_csv(..., sep=';')`.
- In the raw EV file `Start_plugin_hour` is just an **integer hour**, so it won't merge against the
  traffic `Date_from` timestamps. You'll need to build an hour-floored timestamp from `Start_plugin`
  first (Task 3).
- The file still carries non-numeric string columns (`User_type`, `month_plugin`, `weekdays_plugin`)
  and the traffic sensors use `'-'` for missing readings -- handle these in the cleaning steps so
  `.astype(float)` succeeds.
- `models/model4500.pth` was trained on a **26-feature, one-hot-encoded** version of the data. If you
  drop the categorical columns you'll get fewer features and that saved model won't load against your
  data (Task 19). To use it you'd one-hot encode the categoricals instead of dropping them.

In [ ]:
# Setup - import basic data libraries
import numpy as np
import pandas as pd

## Task Group 1 - Load, Inspect, and Merge Datasets

### Task 1
Import `datasets/EV charging reports.csv` into a DataFrame named `ev_charging_reports` and preview
the first five rows with `.head()`. This file holds per-session EV charging data: user/garage info,
plug-in / plug-out times, charging loads, and session dates.

*Hint: this file is semicolon-delimited.*

In [ ]:
# your code here


### Task 2
Import `datasets/Local traffic distribution.csv` into a DataFrame named `traffic_reports` and preview
the first five rows. This dataset gives hourly local traffic-density counts at 5 nearby locations.

In [ ]:
# your code here


### Task 3
Merge the two datasets. The same location can charge at different rates depending on how many cars
are plugged in, so nearby traffic density may help the model. Merge `ev_charging_reports` and
`traffic_reports` into `ev_charging_traffic`, joining `Start_plugin_hour` (charging) to `Date_from`
(traffic).

*Hint: raw `Start_plugin_hour` is just an integer hour. Build an hour-floored timestamp from
`Start_plugin` (format `dd.mm.YYYY HH:00`) so it matches `Date_from` before merging.*

In [ ]:
# your code here


### Task 4
Use `.info()` to inspect the merged dataset, paying attention to data types and missing values.

*What do you notice about the dtypes and missing-value counts?*

In [ ]:
# your code here


## Task Group 2 - Data Cleaning and Preparation

### Task 5
Shrink the dataset by dropping columns you won't train on: ID columns, columns with lots of missing
data, and (for now) non-numeric columns. The project's suggested list is below -- with the raw data
you'll also want to drop the leftover string categoricals (`User_type`, `month_plugin`,
`weekdays_plugin`).

```python
['session_ID', 'Garage_ID', 'User_ID',
 'Shared_ID',
 'Plugin_category', 'Duration_category',
 'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour',
 'Date_from', 'Date_to']
```

In [ ]:
# your code here


### Task 6
`El_kWh` and `Duration_hours` are `object` dtype because the data uses European notation: commas `,`
as the decimal separator instead of periods `.`. Replace `,` with `.` in the affected columns.

*Hint: some traffic columns are also strings with commas, and use `'-'` for missing readings.*

In [ ]:
# your code here


### Task 7
Convert every column of `ev_charging_traffic` to `float`.

*Hint: drop any rows still holding missing values first so the cast succeeds.*

In [ ]:
# your code here


## Task Group 3 - Train Test Split
Split into training data (to fit the model) and testing data (to evaluate it).

### Task 8
Create `X` (input numerical features) and `y` (the target column `El_kWh`).

In [ ]:
# your code here


### Task 9
Use scikit-learn to split `X` and `y` into train/test sets. Use 80% for training and set
`random_state=2`.

In [ ]:
# your code here


## Task Group 4 - Linear Regression Baseline
Optional but useful: if a plain linear regression does just as well, there's no need for a neural
network. Use it as a baseline to beat.

### Task 10
Train a scikit-learn `LinearRegression` on the training data to predict charging loads.

In [ ]:
# your code here


### Task 11
Evaluate the baseline with `mean_squared_error` on the test data. Save it to `test_mse` and print it.

The project quotes ~131.4 for their column set; with the features here you'll get a somewhat different
number. Remember it's squared error -- its square root is the average miss in kWh.

In [ ]:
# your code here


## Task Group 5 - Train a Neural Network Using PyTorch

### Task 12
Import PyTorch: the `torch` library, `nn` (network building blocks and loss functions), and `optim`
(optimizers).

In [ ]:
# your code here


### Task 13
Convert the training and testing sets into PyTorch tensors with `float` values.

*Hint: reshaping `y` to a column vector (`.view(-1, 1)`) makes its shape match the 1-node output.*

In [ ]:
# your code here


### Task 14
Set a random seed with `torch.manual_seed(42)`, then build a `nn.Sequential` network:
input layer sized to the number of features -> hidden layer of 56 nodes + ReLU -> hidden layer of
26 nodes + ReLU -> output layer of 1 node. Save it to `model`.

In [ ]:
# your code here


### Task 15
Define the loss function and optimizer: MSE loss in `loss`, and an Adam optimizer in `optimizer` with
learning rate `0.0007`.

In [ ]:
# your code here


### Task 16
Train for 3000 epochs, printing the training MSE every 500 epochs.

In [ ]:
# your code here


### Task 17
Save the trained network to `models/model.pth`.

In [ ]:
# your code here


### Task 18
Evaluate the network on the test set. Save the test loss to `test_loss` and print it with `.item()`.

*Hint: wrap the forward pass in `torch.no_grad()` so no gradients are tracked.*

In [ ]:
# your code here


### Task 19
Load the longer-trained model at `models/model4500.pth` (trained for 4500 epochs) and evaluate it.

*Note: this saved model expects 26 input features (it was trained on a one-hot-encoded version of the
data). If your pipeline dropped the categorical columns, it won't load against your tensors -- check
`model4500[0].in_features` against `X_test_tensor.shape[1]`. To actually use it, one-hot encode
`month_plugin` / `weekdays_plugin` / `User_type` back in instead of dropping them.*

In [ ]:
# your code here


## Wrap-up
Ideas to keep experimenting once the baseline works:
- Explore different ways to clean and prepare the data (e.g. one-hot encode the categoricals).
- More features don't guarantee a better model -- try different input columns.
- Tune the hidden-layer sizes, activation functions, and learning rate.
- Train for more epochs.